In [ ]:
from utils import load_data, seq_duplicate, seq_generation, extract_features_to_csv, generate_bitcoin_mat, generate_svd_mat, generate_npy_for_fast_inference

from pathlib import Path

# Khi chạy trong Jupyter, cwd() = thư mục chứa notebook = b4e/src/
B4E_DIR = Path.cwd().parent
RAW_DIR         = B4E_DIR / "data_raw"
DATA_STRUCTURED = B4E_DIR / "data_structured"

In [ ]:
print("Add normal account transactions.")
f_in  = open(RAW_DIR / "normal_eoa_transaction_in_slice_1000K.csv", "r")
f_out = open(RAW_DIR / "normal_eoa_transaction_out_slice_1000K.csv", "r")
eoa2seq_in, eoa2seq_out = load_data(f_in, f_out)

eoa2seq_agg = seq_duplicate(eoa2seq_in, eoa2seq_out)
# eoa2seq_agg = seq_generation(eoa2seq_in, eoa2seq_out)

print("Add phishing..")
phisher_f_in  = open(RAW_DIR / "phisher_transaction_in.csv", "r")
phisher_f_out = open(RAW_DIR / "phisher_transaction_out.csv", "r")
phisher_eoa2seq_in, phisher_eoa2seq_out = load_data(phisher_f_in, phisher_f_out)

phisher_eoa2seq_agg = seq_duplicate(phisher_eoa2seq_in, phisher_eoa2seq_out)
# phisher_eoa2seq_agg = seq_generation(phisher_eoa2seq_in, phisher_eoa2seq_out)

eoa2seq_agg.update(phisher_eoa2seq_agg)
print("Total accounts:", len(eoa2seq_agg))

In [ ]:
# df = extract_features_to_csv(eoa2seq_agg, output_filename=str(DATA_STRUCTURED / "b4e_raw_features.csv"))
# df = pd.read_csv("b4e_raw_features.csv")

In [ ]:
generate_bitcoin_mat(eoa2seq_agg, RAW_DIR / "phisher_account.txt", str(DATA_STRUCTURED / "b4e.mat"))


In [ ]:
generate_svd_mat(dataset_str='b4e', emb_dimension=10)

In [ ]:
generate_npy_for_fast_inference(mat_path=DATA_STRUCTURED / 'b4e_svd_10.mat', output_name='b4e', subgraph_size=4)


In [ ]:
import scipy.io as sio

with open(RAW_DIR / "phisher_account.txt") as f:
    phisher_accounts = set(line.strip() for line in f)

print("Total phisher accounts:", len(phisher_accounts))

mat = sio.loadmat(str(DATA_STRUCTURED / "b4e.mat"))
raw_node_ids = mat["node_ids"].squeeze()
node_ids = [nid.item().strip() for nid in raw_node_ids]

label_phish = [1 if nid in phisher_accounts else 0 for nid in node_ids]
print("Phisher nodes found:", sum(label_phish))


In [ ]:
node_ids

In [ ]:
phisher_accounts